In [1]:
!pip install boto3 datasets pyspark==3.4.1
!pip install python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 46.4 MB/s  0:00:06:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pyarrow-24.0.0-cp310-cp310-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached aiohttp-3.14.1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (8.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached async_timeout-5.0.1-py3-none-any.whl.metadata (5.1 kB)
  Using cached frozenlist-1.8.0-cp310-cp310-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl.metadata (20 kB)
  Using cached multidict-6.7.1-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (5.3 kB)
  Using cached propcache-0.5.2-cp310-cp3

In [10]:
from dotenv import dotenv_values

config = dotenv_values(".env")

AWS_ACCESS_KEY_ID = config.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = config.get("AWS_SECRET_ACCESS_KEY")
bucket_name = "mercari-bigdata-project-bucket"
print("AWS credentials loaded from .env file.")

AWS credentials loaded from .env file.


In [11]:
import boto3

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

response = s3.list_objects_v2(Bucket=bucket_name)
if response:
    print("Loaded successfully!")
else:
    print("Failed")

Loaded successfully!


In [12]:
unique_prefixes = set()
if 'Contents' in response:
    for obj in response['Contents']:
        key = obj['Key']
        last_slash_index = key.rfind('/')
        if last_slash_index != -1:
            prefix = key[:last_slash_index + 1]
            unique_prefixes.add(prefix)
        else:
            unique_prefixes.add('/')

    if unique_prefixes:
        print("Unique prefixes found:")
        for i, prefix in enumerate(sorted(list(unique_prefixes))):
            if i < 10: 
                print(f"- {prefix}")
            elif i == 10:
                print("... (truncated, showing first 10 prefixes)")
                break
    else:
        print("No unique prefixes found (or no files in subdirectories).")
else:
    print("No files found in this bucket.")

Unique prefixes found:
- 01_raw/20230501/
- 01_raw/20230601/
- 01_raw/20230701/
- 01_raw/20230801/


In [13]:
import math

def human_readable_size(size_bytes):
    if size_bytes == 0:
        return "0 B"
    size_name = ("B", "KB", "MB", "GB", "TB", "PB", "EB", "ZB", "YB")
    i = int(math.floor(math.log(size_bytes, 1024)))
    p = math.pow(1024, i)
    s = round(size_bytes / p, 2)
    return f"{s} {size_name[i]}"

total_size_bytes = 0
paginator = s3.get_paginator('list_objects_v2')
response_iterator = paginator.paginate(Bucket=bucket_name)

print(f"Calculating total data size in bucket '{bucket_name}'...")

for page in response_iterator:
    if 'Contents' in page:
        for obj in page['Contents']:
            total_size_bytes += obj['Size']

print(f"Total data size in S3 bucket '{bucket_name}': {human_readable_size(total_size_bytes)}")

Calculating total data size in bucket 'mercari-bigdata-project-bucket'...
Total data size in S3 bucket 'mercari-bigdata-project-bucket': 154.67 GB
